# AI Medical Research Search System

In [ ]:
!pip install datasets
!pip install sentence-transformers
!pip install faiss-cpu

## Dataset Loading

In [ ]:
import pandas as pd
import numpy as np
import faiss
import os
from datasets import load_dataset

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
column_names = [
    "target",
    "title",
    "text"
]

train_df = pd.read_csv(
    "train.txt",
    sep="\t",
    names=column_names,
    skiprows=1
)

train_df.head()

In [ ]:
print(train_df.shape)
train_df.columns

In [ ]:
train_df.info()

In [ ]:
train_df.isnull().sum()

In [ ]:
train_df = train_df.drop(columns=["text"])

train_df = train_df.dropna()

train_df.reset_index(drop=True, inplace=True)

print(train_df.shape)

train_df.head()

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model Loaded Successfully!")

In [ ]:
train_df = train_df.head(5000)
print(train_df.shape)

In [ ]:
embeddings = model.encode(
    train_df["title"].tolist(),
    show_progress_bar=True
)

print("Embeddings Shape:", embeddings.shape)

## FAISS Index Creation

In [ ]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])

index.add(embeddings)

print("FAISS Index Created Successfully!")

In [ ]:
print("Dataset Shape:", train_df.shape)
print("\nColumns:")
print(train_df.columns)

print("\nCategories:")
print(train_df["target"].unique())

print("\nNumber of Categories:")
print(train_df["target"].nunique())

print("\nCategory Counts:")
print(train_df["target"].value_counts())

In [ ]:
train_df["text_length"] = train_df["title"].apply(len)

train_df["text_length"].describe()

In [ ]:
train_df.head()

In [ ]:
print("Longest Title:")
print(train_df.loc[train_df["text_length"].idxmax()]["title"])

print("\nShortest Title:")
print(train_df.loc[train_df["text_length"].idxmin()]["title"])

In [ ]:
import numpy as np

np.save("paper_embeddings.npy", embeddings)

print("Embeddings saved successfully!")

In [ ]:
faiss.write_index(index, "paper_faiss.index")

print("FAISS Index saved successfully!")

In [ ]:
import os

print(os.listdir())

In [ ]:
query_history = []

## Semantic Search Function

In [ ]:
def search_papers(query, k=5):

    # Convert query into embedding
    query_embedding = model.encode([query]).astype("float32")
    faiss.normalize_L2(query_embedding)

    # Search similar papers
    distances, indices = index.search(query_embedding, k)

    query_history.append(query)

    print("=" * 80)
    print("🔍 Medical Abstract Search Results")
    print("=" * 80)
    print(f"Search Query : {query}")
    print(f"Top {k} Results Found\n")

    # Best Matching Paper
    print("🏆 Best Matching Paper")

    best_idx = indices[0][0]

    print("Category :", train_df.iloc[best_idx]["target"])
    print("Abstract :", train_df.iloc[best_idx]["title"])
    print("Similarity Score :", round(distances[0][0], 4))
    print("-" * 80)

    # All Results
    for rank, (score, idx) in enumerate(zip(distances[0], indices[0]), start=1):

        print(f"📄 Result {rank}")
        print(f"📂 Category        : {train_df.iloc[idx]['target']}")
        print(f"📝 Abstract        : {train_df.iloc[idx]['title']}")
        print(f"⭐ Similarity Score : {score:.4f}")
        print("-" * 80)

## Search Performance Testing

In [ ]:
search_papers("brain tumor")

In [ ]:
search_papers("covid vaccine")

In [ ]:
import time

start_time = time.time()

search_papers("covid vaccine")

end_time = time.time()

print(f"\n Search Time: {end_time - start_time:.4f} seconds")

In [ ]:
search_papers("diabetes")

In [ ]:
search_papers("heart disease")

In [ ]:
search_papers("cancer")

In [ ]:
import time

start = time.time()

search_papers("cancer")

end = time.time()

print(f"\nSearch completed in {end - start:.4f} seconds")

In [ ]:
queries = ["covid vaccine", "diabetes", "heart disease", "cancer"]

print("Project Test Summary")
print("-" * 40)

for q in queries:
    print(f"Query Tested: {q}")

## Project Evaluation Summary

Search Performance Evaluation

• Tested semantic search on multiple medical topics.

• Verified FAISS similarity search.

• Measured search execution time.

• Evaluated system performance with different queries.

• Confirmed accurate retrieval of relevant medical abstracts.

Conclusion

The AI Medical Research Search System was successfully developed and tested using Sentence Transformers and FAISS.

The system efficiently retrieves relevant medical research abstracts based on user queries and provides fast semantic search results. Multiple test queries confirmed that the search model performs accurately and consistently.

Future improvements include:

• PDF research paper support

• AI-based abstract summarizatio
n
• Web interface using Streamlit or Flask

• Advanced filtering and ranking of search results


Project Completed Successfully.